# 2026 인하 AI 챌린지 — 로봇 미래 행동 영상 생성 파이프라인

**평가 산식**: Score = 0.3×DINO + 0.3×VideoFeature + 0.4×Action  (낮을수록 좋음)

**파이프라인 흐름**
```
eval/images/*.png + eval/actions/*.npy
    → [영상 생성 모델]
    → submission_kit/input_videos/sample_XXXXXX.mp4
    → make_submission_csv.py
    → submission_features.csv  (데이콘 제출)
```

## 0. 경로 및 GPU 설정

In [ ]:
import os
import sys
from pathlib import Path

# 이 노트북은 /home1/sota/inha2026/ 에서 실행합니다.
PKG_ROOT = Path('/home1/sota/inha2026').resolve()
BASELINE_ROOT = PKG_ROOT / 'baseline'
CHALLENGE_KIT = BASELINE_ROOT / 'challenge_kit'
DATA_ROOT = PKG_ROOT / 'data'
EVAL_ROOT = DATA_ROOT / 'eval'
TRAIN_ROOT = DATA_ROOT / 'train'
SUBMISSION_KIT = PKG_ROOT / 'submission_kit'
INPUT_VIDEOS = SUBMISSION_KIT / 'input_videos'
ACTION_STATS = TRAIN_ROOT / 'so100_action_statistics.json'
BASELINE_CKPT = BASELINE_ROOT / 'checkpoints' / 'baseline_diffusion.ckpt'
BACKBONE_CKPT = BASELINE_ROOT / 'checkpoints' / 'backbone.ckpt'

print('PKG_ROOT      :', PKG_ROOT)
print('eval images   :', len(list((EVAL_ROOT / 'images').glob('*.png'))))
print('eval actions  :', len(list((EVAL_ROOT / 'actions').glob('*.npy'))))
print('train exists  :', TRAIN_ROOT.exists())
print('baseline ckpt :', BASELINE_CKPT.exists())
print('backbone ckpt :', BACKBONE_CKPT.exists())

In [ ]:
import torch

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))
    print('VRAM  :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. 데이터 탐색

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# action 통계
with open(ACTION_STATS) as f:
    stats = json.load(f)

ACTION_NAMES = ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']
print(f"전체 학습 프레임 수: {stats['count']:,}")
print("\nAction 통계 (mean / std):")
for i, name in enumerate(ACTION_NAMES):
    print(f"  {name:16s}: mean={stats['mean'][i]:8.3f}, std={stats['std'][i]:8.3f}")

In [ ]:
# 학습 데이터 구성 파악
users = sorted(p.name for p in TRAIN_ROOT.iterdir() if p.is_dir() and p.name != '__pycache__')
print(f"제공 사용자(contributor) 수: {len(users)}")

datasets = []
for user in users:
    for ds in (TRAIN_ROOT / user).iterdir():
        if ds.is_dir():
            meta_path = ds / 'meta' / 'info.json'
            if meta_path.exists():
                with open(meta_path) as f:
                    info = json.load(f)
                datasets.append({'user': user, 'dataset': ds.name, **{k: info.get(k) for k in ['total_episodes', 'total_frames', 'fps']}})

import pandas as pd
df_train = pd.DataFrame(datasets)
print(f"총 데이터셋 수: {len(df_train)}")
print(f"총 에피소드 수: {df_train['total_episodes'].sum():,}")
print(f"총 프레임 수  : {df_train['total_frames'].sum():,}")
df_train.head(10)

In [ ]:
# eval 샘플 시각화 (첫 6개)
eval_images = sorted((EVAL_ROOT / 'images').glob('*.png'))[:6]
eval_actions = sorted((EVAL_ROOT / 'actions').glob('*.npy'))[:6]

fig, axes = plt.subplots(2, 6, figsize=(20, 6))
for i, (img_path, act_path) in enumerate(zip(eval_images, eval_actions)):
    img = Image.open(img_path)
    act = np.load(act_path)  # (16, 6)
    axes[0, i].imshow(img)
    axes[0, i].set_title(img_path.stem, fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].plot(act)  # 16 steps, 6 dims
    axes[1, i].set_title('action seq', fontsize=8)
    if i == 0:
        axes[1, i].legend(ACTION_NAMES, fontsize=5, loc='upper right')

plt.tight_layout()
plt.savefig(PKG_ROOT / 'eval_samples_preview.png', dpi=100)
plt.show()
print('shape of action file:', np.load(eval_actions[0]).shape)

In [ ]:
# eval action 전체 분포 확인
all_actions = np.stack([np.load(p) for p in sorted((EVAL_ROOT / 'actions').glob('*.npy'))])  # (216, 16, 6)
print('eval action tensor shape:', all_actions.shape)

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for i, (ax, name) in enumerate(zip(axes.flat, ACTION_NAMES)):
    vals = all_actions[:, :, i].flatten()
    ax.hist(vals, bins=50)
    ax.set_title(name)
    ax.axvline(vals.mean(), color='red', linestyle='--', label=f'mean={vals.mean():.1f}')
    ax.legend()

plt.suptitle('Eval Action Distribution (216 samples × 16 steps)')
plt.tight_layout()
plt.savefig(PKG_ROOT / 'eval_action_distribution.png', dpi=100)
plt.show()

## 2. 패키지 설치

In [ ]:
import subprocess, sys

# baseline 의존 패키지 설치
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BASELINE_ROOT / 'requirements.txt')])

# challenge_kit, dynamicrafter, video_utils 설치
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    '-e', str(CHALLENGE_KIT),
    '-e', str(CHALLENGE_KIT / 'libs' / 'dynamicrafter'),
    '-e', str(BASELINE_ROOT / 'shared_libs' / 'video_utils'),
])

# submission_kit 의존 패키지 설치
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SUBMISSION_KIT / 'requirements.txt')])

print('설치 완료')

## 3. 베이스라인 Backbone 준비

DynamiCrafter 사전학습 가중치(backbone.ckpt)가 필요합니다.  
`baseline_diffusion.ckpt`는 이미 `baseline/checkpoints/`에 포함되어 있습니다.

In [ ]:
from urllib.request import urlretrieve

BACKBONE_URL = 'https://huggingface.co/Doubiiu/DynamiCrafter_512/resolve/main/model.ckpt'

if BACKBONE_CKPT.exists():
    print(f'이미 존재: {BACKBONE_CKPT}')
else:
    print(f'다운로드 중: {BACKBONE_URL}')
    BACKBONE_CKPT.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(BACKBONE_URL, BACKBONE_CKPT)
    print(f'완료: {BACKBONE_CKPT}')

print(f'baseline ckpt: {BASELINE_CKPT.exists()} ({BASELINE_CKPT})')
print(f'backbone ckpt: {BACKBONE_CKPT.exists()} ({BACKBONE_CKPT})')

## 4. 베이스라인 영상 생성 (추론)

DynamiCrafter + `baseline_diffusion.ckpt`로 eval 216개 샘플에 대한 mp4를 생성합니다.  
`submission_kit/input_videos/sample_XXXXXX.mp4` 형식으로 저장됩니다.

In [ ]:
# [선택] 커스텀 checkpoint를 사용하려면 아래 경로를 변경하세요.
INFERENCE_CKPT = str(BASELINE_CKPT)   # 베이스라인 기본값
# INFERENCE_CKPT = str(PKG_ROOT / 'outputs' / 'my_finetuned.ckpt')  # 파인튜닝 이후

INPUT_VIDEOS.mkdir(parents=True, exist_ok=True)
already = len(list(INPUT_VIDEOS.glob('*.mp4')))
print(f'이미 생성된 mp4: {already} / 216')

In [ ]:
import subprocess, sys, os

env = os.environ.copy()
env['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

result = subprocess.run([
    sys.executable,
    'scripts/inference/generate_baseline_videos.py',
    '--checkpoint', INFERENCE_CKPT,
    '--challenge-root', str(EVAL_ROOT),
    '--prediction-root', str(INPUT_VIDEOS),
    '--action-stats-path', str(ACTION_STATS),
    # '--ddim-steps', '30',      # 기본 50. 속도 우선 시 줄일 수 있음
    # '--overwrite',             # 이미 생성된 파일도 재생성
], cwd=str(CHALLENGE_KIT), env=env)

generated = len(list(INPUT_VIDEOS.glob('*.mp4')))
print(f'\n생성 완료: {generated} / 216 mp4')

In [ ]:
# 생성된 영상 하나 확인
from IPython.display import Video, display

sample_mp4 = list(INPUT_VIDEOS.glob('*.mp4'))
if sample_mp4:
    display(Video(str(sorted(sample_mp4)[0]), width=512))
else:
    print('생성된 mp4 없음')

## 5. 커스텀 모델 개발 (선택)

베이스라인을 기반으로 성능을 개선하려면 이 섹션에서 파인튜닝을 진행합니다.

### 5-1. 학습 실행

**학습 시간 제한**: RTX PRO 6000 (96GB) 기준 최대 4일  
**학습 config**: `baseline/challenge_kit/configs/train/inha_action_diffusion_11M.yaml`

아래 셀에서 `RUN_TRAINING = True`로 바꾸면 학습이 시작됩니다.

In [ ]:
RUN_TRAINING = False  # True로 변경 시 학습 시작
TRAIN_CONFIG = 'configs/train/inha_action_diffusion_11M.yaml'

if RUN_TRAINING:
    import shutil, os
    if shutil.which('bash') is None:
        raise RuntimeError('bash가 필요합니다.')
    env = os.environ.copy()
    env['PYTHON_BIN'] = sys.executable
    env['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
    subprocess.check_call([
        'bash', 'scripts/train.sh',
        '--config', TRAIN_CONFIG,
        '--script', 'scripts/train_diffusion.py',
    ], cwd=str(CHALLENGE_KIT), env=env)
else:
    print('학습 건너뜀 (RUN_TRAINING=False). 학습하려면 True로 변경하세요.')

### 5-2. 학습 결과 확인

In [ ]:
# 학습 완료 후 최신 체크포인트 확인
output_dir = BASELINE_ROOT / 'outputs' / 'diffusion'
if output_dir.exists():
    ckpts = sorted(output_dir.rglob('*.ckpt'))
    print(f'저장된 체크포인트 ({len(ckpts)}개):')
    for c in ckpts[-5:]:
        print(' ', c)
else:
    print('학습 결과 없음:', output_dir)

## 6. 제출 CSV 생성

`submission_kit/input_videos/` 안의 mp4를 DINO / VideoFeature / Action feature로 변환합니다.  
**규정**: `make_submission_csv.py` 출력 CSV를 수동으로 수정하면 실격입니다.

In [ ]:
# 모든 216개 mp4가 준비됐는지 확인
SAMPLE_IDS = sorted(p.stem for p in (EVAL_ROOT / 'images').glob('*.png'))
missing = [sid for sid in SAMPLE_IDS if not (INPUT_VIDEOS / f'{sid}.mp4').exists()]
if missing:
    print(f'[경고] {len(missing)}개 mp4 없음: {missing[:5]} ...')
else:
    print(f'모든 {len(SAMPLE_IDS)}개 mp4 준비 완료')

In [ ]:
import subprocess, sys, os

result = subprocess.run([
    sys.executable, 'make_submission_csv.py',
    '--challenge-root', str(EVAL_ROOT),
    '--action-stats-path', str(ACTION_STATS),
], cwd=str(SUBMISSION_KIT))

output_csv = SUBMISSION_KIT / 'submission_features.csv'
print('\n제출 CSV 생성 완료:', output_csv.exists())

## 7. 제출 파일 검증

In [ ]:
import pandas as pd

output_csv = SUBMISSION_KIT / 'submission_features.csv'
df = pd.read_csv(output_csv)

print('shape:', df.shape)  # 기대값: (648, 3) = 216샘플 × 3종 feature
print('columns:', list(df.columns))
print('\nfeature_component 분포:')
print(df['feature_component'].value_counts())
print('\n결측치:', df.isnull().sum().sum())
df.head()

In [ ]:
# 제출 유효성 최종 점검
assert df.shape == (648, 3), f"예상 (648, 3), 실제 {df.shape}"
assert set(df['feature_component'].unique()) == {
    'DINO Component', 'Video Feature Component', 'Action Component'
}, "feature_component 값 오류"
assert df['feature_json'].notna().all(), "feature_json 결측치 존재"
assert set(df['sample_id'].unique()) == set(SAMPLE_IDS), "sample_id 불일치"

print('✓ 유효성 검사 통과')
print(f'→ 데이콘에 제출할 파일: {output_csv}')

---
## 부록: 개선 아이디어

| 전략 | 방법 | 예상 효과 |
|------|------|-----------|
| **데이터 증강** | 학습 시 color jitter, horizontal flip(좌우 대칭 시 action도 반전) | DINO/Video Feature 향상 |
| **Action 가중치** | Action Component가 40% 비중 → action conditioning 강화 (dropout 0.0 유지) | Action 점수 ↓ |
| **DDIM steps 증가** | 50 → 100 스텝, CFG scale 조정 | 품질 향상 |
| **해상도 조정** | 320×512 → 256×512 or 더 큰 해상도 실험 | DINO/Video |
| **앙상블** | 여러 checkpoint의 결과를 평균하거나 선택 | 안정성 |
| **Temporal 길이** | 16 frames → 더 긴 시퀀스 실험 | Video Feature |

> **주의**: eval 데이터는 어떤 형태로도 학습에 사용 불가 (Data Leakage 실격)